# **1. Data processing**

This notebook organizes the observations at two levels: player–match and player–league–season–round. Round-level reference statistics are calculated with **all observed players** before the target age range is selected.

League changes remain separated by `competition_id` and are identified through `league_spell_number`, preserving the sequence of each player's competitive context.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
MIN_AGE, MAX_AGE = 16, 23
COMPETITION_CODE_BY_SOURCE = {
    'GB1': 'ENG-1', 'ES1': 'SPA-1', 'L1': 'GER-1', 'IT1': 'ITA-1',
    'FR1': 'FRA-1', 'PO1': 'POR-1', 'BRA1': 'BR-1', 'NL1': 'NED-1',
    'BE1': 'BEL-1', 'AR1N': 'ARG-1', 'ARG1': 'ARG-1',
    'ARGC': 'ARG-1', 'COLP': 'COL-1', 'COL1': 'COL-1',
    'COLF': 'COL-1', 'PR1A': 'PAR-1', 'PR1C': 'PAR-1',
    'URU1': 'URU-1', 'URUI': 'URU-1', 'URUC': 'URU-1', 'SA1': 'KSA-1',
    'MEXA': 'MEX-1', 'MEX1': 'MEX-1',
    'MLS1': 'USA-1',
}
ROUND_KEYS = ['competition_id', 'source_competition_id', 'season_id', 'round_id']
PLAYER_ROUND_KEYS = ['player_id', *ROUND_KEYS]

## **2. Transformation design**

Match timestamps, minutes, and identifiers are normalized before duplicate player–match records are removed. Exact age is calculated on each match date, and player–round observations retain the underlying match and club identifiers for auditability.

In [2]:
def pipe_unique(values):
    return '|'.join(sorted({str(value) for value in values.dropna()}))

def build_tables(matches, players, min_age=18, max_age=23):
    required_matches = {'player_id', 'game_id', 'season_id', 'round_id',
                        'match_date_utc', 'club_id', 'minutes_played'}
    required_players = {'player_id', 'birth_date'}
    if missing := required_matches - set(matches):
        raise ValueError(f'Missing columns in player_matches: {sorted(missing)}')
    if missing := required_players - set(players):
        raise ValueError(f'Missing columns in players: {sorted(missing)}')

    frame = matches.copy()
    if 'source_competition_id' not in frame and 'competition_id' in frame:
        frame = frame.rename(columns={'competition_id': 'source_competition_id'})
    if 'source_competition_id' not in frame:
        raise ValueError('Missing source_competition_id in player_matches')
    frame['source_competition_id'] = frame['source_competition_id'].astype(str)
    mapped_codes = frame['source_competition_id'].map(COMPETITION_CODE_BY_SOURCE)
    domestic_league_mask = mapped_codes.notna()
    excluded_non_league_rows = int((~domestic_league_mask).sum())
    frame = frame.loc[domestic_league_mask].copy()
    frame['competition_id'] = mapped_codes.loc[domestic_league_mask]
    domestic_rows_before_deduplication = len(frame)
    for column in ['player_id', 'game_id', 'competition_id']:
        frame[column] = frame[column].astype(str)
    frame['season_id'] = pd.to_numeric(frame['season_id'], errors='coerce').astype('Int64')
    frame['round_id'] = pd.to_numeric(frame['round_id'], errors='coerce').astype('Int64')
    frame['match_timestamp_utc'] = pd.to_datetime(frame['match_date_utc'], errors='coerce', utc=True)
    frame['minutes_played'] = pd.to_numeric(frame['minutes_played'], errors='coerce').fillna(0).clip(lower=0)
    frame['match_duration'] = pd.to_numeric(frame.get('match_duration'), errors='coerce')
    frame = (frame.sort_values(['player_id', 'match_timestamp_utc', 'game_id'], na_position='last')
                  .drop_duplicates(['player_id', 'game_id'], keep='last'))

    people = players.copy()
    people['player_id'] = people['player_id'].astype(str)
    people['birth_date'] = pd.to_datetime(people['birth_date'], errors='coerce').dt.normalize()
    people = people[['player_id', 'birth_date']].drop_duplicates('player_id', keep='last')
    frame = frame.merge(people, on='player_id', how='left', validate='many_to_one')
    match_day = frame['match_timestamp_utc'].dt.tz_localize(None).dt.normalize()
    frame['age_days'] = (match_day - frame['birth_date']).dt.days.astype('Int64')
    frame['age_decimal'] = frame['age_days'] / 365.2425
    frame['age_years'] = np.floor(frame['age_decimal']).astype('Int64')
    frame['played'] = frame['minutes_played'].gt(0)
    frame['minutes_share_match'] = np.where(
        frame['match_duration'].gt(0), frame['minutes_played'] / frame['match_duration'], np.nan
    )

    valid = frame.dropna(subset=['season_id', 'round_id']).copy()
    valid = valid.sort_values(['player_id', 'match_timestamp_utc', 'game_id'])
    rounds = (valid.groupby(PLAYER_ROUND_KEYS, as_index=False, observed=True)
        .agg(first_match_timestamp_utc=('match_timestamp_utc', 'min'),
             last_match_timestamp_utc=('match_timestamp_utc', 'max'),
             n_match_records=('game_id', 'nunique'), game_ids=('game_id', pipe_unique),
             club_ids=('club_id', pipe_unique), minutes_played=('minutes_played', 'sum'),
             available_match_minutes=('match_duration', 'sum'), played_any=('played', 'max'),
             birth_date=('birth_date', 'first'), age_days=('age_days', 'first'),
             age_decimal=('age_decimal', 'first'), age_years=('age_years', 'first'))
        .sort_values([*ROUND_KEYS, 'player_id']).reset_index(drop=True))
    rounds['minutes_share_available'] = np.where(
        rounds['available_match_minutes'].gt(0),
        rounds['minutes_played'] / rounds['available_match_minutes'], np.nan
    )

    reference = (rounds.groupby(ROUND_KEYS, as_index=False, observed=True)
        .agg(n_players_observed=('player_id', 'nunique'), n_players_played=('played_any', 'sum'),
             total_minutes=('minutes_played', 'sum'),
             mean_minutes_all_players=('minutes_played', 'mean'),
             median_minutes_all_players=('minutes_played', 'median'),
             std_minutes_all_players=('minutes_played', lambda x: x.std(ddof=0)),
             q25_minutes_all_players=('minutes_played', lambda x: x.quantile(.25)),
             q75_minutes_all_players=('minutes_played', lambda x: x.quantile(.75)),
             n_players_with_known_age=('age_years', 'count'))
        .sort_values(ROUND_KEYS).reset_index(drop=True))
    reference['share_players_played'] = reference['n_players_played'] / reference['n_players_observed']

    rounds = rounds.merge(reference, on=ROUND_KEYS, how='left', validate='many_to_one')
    rounds['minutes_vs_round_mean_all'] = np.where(
        rounds['mean_minutes_all_players'].gt(0),
        rounds['minutes_played'] / rounds['mean_minutes_all_players'], np.nan
    )
    rounds['minutes_zscore_round_all'] = np.where(
        rounds['std_minutes_all_players'].gt(0),
        (rounds['minutes_played'] - rounds['mean_minutes_all_players']) / rounds['std_minutes_all_players'],
        np.nan
    )
    rounds['minutes_percentile_round_all'] = rounds.groupby(ROUND_KEYS, observed=True)['minutes_played'].rank(
        method='average', pct=True
    )

    target = rounds.loc[rounds['age_years'].between(min_age, max_age, inclusive='both')].copy()
    trajectories = target.sort_values(
        ['player_id', 'first_match_timestamp_utc', 'competition_id', 'round_id']
    ).reset_index(drop=True)
    trajectories['player_observation_number'] = trajectories.groupby('player_id').cumcount() + 1
    trajectories['league_observation_number'] = trajectories.groupby(
        ['player_id', 'competition_id']).cumcount() + 1
    previous = trajectories.groupby('player_id')['competition_id'].shift()
    new_spell = previous.isna() | trajectories['competition_id'].ne(previous)
    trajectories['league_spell_number'] = new_spell.astype(int).groupby(trajectories['player_id']).cumsum()

    audit = {
        'raw_player_match_rows': int(len(matches)),
        'canonical_player_match_rows': int(len(frame)),
        'excluded_non_domestic_league_rows': excluded_non_league_rows,
        'duplicate_rows_removed': int(domestic_rows_before_deduplication - len(frame)),
        'rows_missing_match_date': int(frame['match_timestamp_utc'].isna().sum()),
        'rows_missing_round': int(frame['round_id'].isna().sum()),
        'rows_missing_birth_date': int(frame['birth_date'].isna().sum()),
        'all_age_player_round_rows': int(len(rounds)),
        'age_16_23_player_round_rows': int(len(target)),
        'age_16_23_players': int(target['player_id'].nunique()),
        'players_in_multiple_leagues': int(target.groupby('player_id')['competition_id'].nunique().gt(1).sum()),
        'normalization_population': 'all observed players before age filtering',
    }
    return frame, rounds, reference, target, trajectories, audit

## **3. Processing and export**

The canonical player–match table is aggregated by league, season, and round. Population benchmarks use every observed age; the 16–23 subset is created only after normalization. All outputs are written as Parquet files with a compact processing audit.

In [3]:
matches_path = RAW / 'player_matches.parquet'
players_path = RAW / 'players.parquet'
if not matches_path.exists() or not players_path.exists():
    raise FileNotFoundError('Run 00_data_collection.ipynb before this notebook.')

matches = pd.read_parquet(matches_path)
players = pd.read_parquet(players_path)
player_match, all_rounds, reference, target_rounds, trajectories, audit = build_tables(
    matches, players, MIN_AGE, MAX_AGE
)

outputs = {
    'player_match_minutes.parquet': player_match,
    'all_age_player_round.parquet': all_rounds,
    'league_round_reference.parquet': reference,
    'age_16_23_player_round.parquet': target_rounds,
    'age_16_23_trajectories.parquet': trajectories,
}
for name, frame in outputs.items():
    frame.to_parquet(PROCESSED / name, index=False)
(PROCESSED / 'processing_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False))
audit

{'raw_player_match_rows': 2338262,
 'canonical_player_match_rows': 2338262,
 'excluded_non_domestic_league_rows': 0,
 'duplicate_rows_removed': 0,
 'rows_missing_match_date': 0,
 'rows_missing_round': 0,
 'rows_missing_birth_date': 658,
 'all_age_player_round_rows': 2326008,
 'age_16_23_player_round_rows': 978164,
 'age_16_23_players': 18772,
 'players_in_multiple_leagues': 2488,
 'normalization_population': 'all observed players before age filtering'}

## **4. Output audit**

The final displays verify round-level reference statistics, ordered player trajectories, and players observed across multiple competitions.

In [4]:
display(reference.head())
display(trajectories.head())
display(trajectories.groupby('player_id')['competition_id'].nunique().sort_values(ascending=False).head(20))

,competition_id,source_competition_id,season_id,round_id,n_players_observed,n_players_played,total_minutes,mean_minutes_all_players,median_minutes_all_players,std_minutes_all_players,q25_minutes_all_players,q75_minutes_all_players,n_players_with_known_age,share_players_played
0,ARG-1,AR1N,2019,1,682,335,23727,34.790323,0.0,40.528673,0.0,90.0,682,0.491202
1,ARG-1,AR1N,2019,2,685,335,23753,34.675912,0.0,40.780847,0.0,90.0,685,0.489051
2,ARG-1,AR1N,2019,3,680,335,23679,34.822059,0.0,40.545390,0.0,90.0,680,0.492647
3,ARG-1,AR1N,2019,4,678,335,23678,34.923304,0.0,40.789481,0.0,90.0,678,0.494100
4,ARG-1,AR1N,2019,5,677,335,23700,35.007386,0.0,40.885237,0.0,90.0,677,0.494830


,player_id,competition_id,source_competition_id,season_id,round_id,first_match_timestamp_utc,last_match_timestamp_utc,n_match_records,game_ids,club_ids,...,q25_minutes_all_players,q75_minutes_all_players,n_players_with_known_age,share_players_played,minutes_vs_round_mean_all,minutes_zscore_round_all,minutes_percentile_round_all,player_observation_number,league_observation_number,league_spell_number
0,1000135,SPA-1,ES1,2023,12,2023-11-05 15:15:00+00:00,2023-11-05 15:15:00+00:00,1,4113168,1049,...,0.0,83.00,595,0.517647,0.0,-0.848538,0.242017,1,1,1
1,1000135,SPA-1,ES1,2023,13,2023-11-11 20:00:00+00:00,2023-11-11 20:00:00+00:00,1,4113096,1049,...,0.0,83.75,598,0.516722,0.0,-0.844525,0.242475,2,2,1
2,1000135,SPA-1,ES1,2023,14,2023-11-25 15:15:00+00:00,2023-11-25 15:15:00+00:00,1,4113094,1049,...,0.0,81.00,596,0.535235,0.0,-0.852556,0.233221,3,3,1
3,1000135,SPA-1,ES1,2023,15,2023-12-02 13:00:00+00:00,2023-12-02 13:00:00+00:00,1,4113135,1049,...,0.0,83.25,600,0.516667,0.0,-0.834030,0.242500,4,4,1
4,1000135,SPA-1,ES1,2023,16,2023-12-08 20:00:00+00:00,2023-12-08 20:00:00+00:00,1,4112878,1049,...,0.0,85.00,613,0.492659,0.0,-0.810376,0.254486,5,5,1


player_id
505653     6
805714     5
649452     5
649317     5
994536     5
668267     4
910905     4
398073     4
613201     4
1145504    4
748319     4
591193     4
688593     4
559328     4
1018259    4
668265     4
368887     4
888493     4
849410     4
926694     4
Name: competition_id, dtype: int64